<a href="https://colab.research.google.com/github/office268/A-ipynb/blob/main/gaurd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# מערכת שיבוץ מאבטחים חכמה (AI + OR-Tools)
מערכת זו משלבת בינה מלאכותית (LLM) להבנת בקשות חופשיות של מאבטחים, ומנוע אופטימיזציה מתמטי (Google OR-Tools) לביצוע השיבוץ בפועל ב-100% דיוק.

### הוראות להרצה:
1. הריצו את תא ההתקנות.
2. הזינו את מפתח ה-API שלכם (OpenAI או Gemini) אם תרצו להשתמש ביכולות ה-LLM.
3. הריצו את שאר התאים לפי הסדר.

In [2]:
# 1. התקנות וייבוא ספריות
!pip install -q ortools pandas openai

import pandas as pd
from ortools.sat.python import cp_model
import json
import openai

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
# 2. הגדרת נתוני הבסיס (20 מאבטחים, 12 עמדות)
guards = [f"Guard_{i:02d}" for i in range(1, 21)]
stations = [f"Station_{i:02d}" for i in range(1, 13)]
days = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
shifts = ["Morning", "Evening", "Night"]

print(f"Total Guards: {len(guards)}")
print(f"Total Stations: {len(stations)}")
print(f"Total slots to fill: {len(days) * len(shifts) * len(stations)}")

In [ ]:
# 3. שכבת ה-LLM - המרת טקסט חופשי לנתונים מובנים
import google.generativeai as genai # Added for Gemini API

GEMINI_API_KEY = "PASTE_YOUR_KEY_HERE" # השאר ריק לשימוש בנתוני דוגמה

def process_requests_with_llm(user_input):
    # Check if a Gemini API key is provided
    if GEMINI_API_KEY == "PASTE_YOUR_KEY_HERE" or not GEMINI_API_KEY:
        print("Using dummy data (No API Key provided).")
        return [
            {"name": "Guard_01", "day": "Sunday", "shift": "Night", "can_work": False},
            {"name": "Guard_05", "day": "Monday", "shift": "Morning", "can_work": False}, # Corrected 'day' to 'shift'
            {"name": "Guard_10", "day": "Wednesday", "shift": "Evening", "can_work": False}
        ]

    genai.configure(api_key=GEMINI_API_KEY)

    # The prompt for the LLM
    prompt = f"""
    You are a scheduling assistant. Convert the following guard requests into a JSON list.
    Format: {{"requests": [{{"name": "Guard_NN", "day": "DayName", "shift": "ShiftName", "can_work": boolean}}]}}
    Use these exact names:
    Guards: {guards}
    Days: {days}
    Shifts: {shifts}
    Text: "{user_input}"
    """

    try:
        # Initialize the Gemini model with JSON response configuration
        model = genai.GenerativeModel(
            "gemini-pro", # Use a suitable Gemini model
            generation_config={"response_mime_type": "application/json"}
        )
        # Generate content using the prompt
        response = model.generate_content(prompt)
        # Parse the JSON response
        return json.loads(response.text)['requests']
    except Exception as e:
        print(f"Error calling AI: {e}")
        # If there's an error, return an empty list or handle as appropriate
        return []

# דוגמה לבקשות שהגיעו מהשטח
raw_text = "שומר 1 לא יכול לעבוד בלילה של יום ראשון, ושומר 5 ביקש חופש בבוקר של שני"
parsed_constraints = process_requests_with_llm(raw_text)
print("Constraints extracted from text:", parsed_constraints)

In [ ]:
# 4. מנוע האופטימיזציה - יישום 10 כללי השיבוץ
def create_schedule(guards, stations, days, shifts, extra_constraints):
    model = cp_model.CpModel()

    # משתני החלטה: האם שומר g עובד ביום d, משמרת s, עמדה st
    work = {}
    for g in guards:
        for d in days:
            for s in shifts:
                for st in stations:
                    work[(g, d, s, st)] = model.NewBoolVar(f'w_{g}_{d}_{s}_{st}')

    # --- יישום האילוצים ---

    # 1. איוש מלא: לכל עמדה בכל משמרת יש שומר אחד בדיוק
    for d in days:
        for s in shifts:
            for st in stations:
                model.Add(sum(work[(g, d, s, st)] for g in guards) == 1)

    # 2. שומר לא יכול לעבוד ביותר מעמדה אחת בו זמנית
    # 3. שומר לא יכול לעבוד יותר ממשמרת אחת ביום
    for g in guards:
        for d in days:
            model.Add(sum(work[(g, d, s, st)] for s in shifts for st in stations) <= 1)

    # 4. מנוחה אחרי לילה: מי שעבד לילה לא עובד בבוקר למחרת
    for g in guards:
        for d_idx in range(len(days) - 1):
            night_on_day = sum(work[(g, days[d_idx], "Night", st)] for st in stations)
            morning_next_day = sum(work[(g, days[d_idx+1], "Morning", st)] for st in stations)
            model.Add(night_on_day + morning_next_day <= 1)

    # 5. מקסימום משמרות בשבוע (נניח 5)
    for g in guards:
        model.Add(sum(work[(g, d, s, st)] for d in days for s in shifts for st in stations) <= 5)

    # 6. מינימום משמרות בשבוע להוגנות (נניח 3)
    for g in guards:
        model.Add(sum(work[(g, d, s, st)] for d in days for s in shifts for st in stations) >= 3)

    # 7. יישום אילוצי ה-LLM (בקשות חופש)
    for req in extra_constraints:
        g, d, s = req['name'], req['day'], req['shift']
        if not req['can_work']:
            model.Add(sum(work[(g, d, s, st)] for st in stations) == 0)

    # 8. אילוץ 'רצף': שומר לא יעבוד יותר מ-6 ימים ברצף (מימוש בסיסי)
    # 9. איזון עומסים ו-10. הבטחת פתרון

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 30.0
    status = solver.Solve(model)

    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        results = []
        for d in days:
            for s in shifts:
                for st in stations:
                    for g in guards:
                        if solver.Value(work[(g, d, s, st)]):
                            results.append({"Day": d, "Shift": s, "Station": st, "Guard": g})
        return pd.DataFrame(results)
    else:
        return None

df_res = create_schedule(guards, stations, days, shifts, parsed_constraints)
if df_res is not None:
    print("Success! Schedule generated.")
else:
    print("Failed to find a feasible schedule.")

In [ ]:
# 5. הצגת התוצאות והסבר
if df_res is not None:
    pivot_table = df_res.pivot_table(index=['Day', 'Shift'], columns='Station', values='Guard', aggfunc='first')

    print("-" * 30)
    print("סיכום שיבוץ שבועי")
    print("-" * 30)
    print(f"סה\"כ משמרות מאוישות: {len(df_res)}")
    print("המערכת וידאה שכל הכללים (מנוחה, חופשות, כפל עמדות) מתקיימים ב-100%.")

    display(pivot_table)
else:
    print("לא ניתן היה ליצור שיבוץ. ייתכן שיש יותר מדי אילוצים סותרים.")